<a href="https://colab.research.google.com/github/andiandiandika3-prog/Mechine-Learning/blob/main/1_Random%20Forest/Model_Umum_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Secara umum Random Forest merupakan pengembangan dari Bagging, yaitu secara berkali-kali melakukan bootstrap terhadap data training dan menyusun pohon klasifikasi berdasarkan data hasil resampling tersebut, dan kemudian proses prediksi dilakukan dengan mengagregasi hasil prediksi dari banyak pohon yang umumnya menggunakan pendekatan majority vote.

# Algoritma dasar dari random forest adalah sebagai berikut:
1.  Given a training data set
2.  Select number of trees to build (n_trees)
3.  for i = 1 to n_trees do
4.  |  Generate a bootstrap sample of the original data
5.  |  Grow a classification tree to the bootstrapped data
6.  |  for each split do
7.  |  | Select m_try variables at random from all p variables
8.  |  | Pick the best variable/split-point among the m_try
9.  |  | Split the node into two child nodes
10. |  end
11. | Use typical tree model stopping criteria to determine when a
    | tree is complete (but do not prune)
12. end
13. Output ensemble of trees

# Template untuk download data dari gdrive (csv)

https://drive.google.com/uc?export=download&id=1X_G95k-GmbDhgC1bH92Oj8-STatG8iDy


# Muat data

In [14]:
df <- read.csv("https://drive.google.com/uc?export=download&id=1X_G95k-GmbDhgC1bH92Oj8-STatG8iDy")

cat("-----------------------------------------","\n")
cat("cek struktur data","\n")
cat("-----------------------------------------","\n")
str(df)

cat("-----------------------------------------","\n")
cat("Tampilkan 5 Data Awal","\n")
cat("-----------------------------------------","\n")
head(df)


cat("----------------------------------------------------","\n")
cat("Membuat variabel target quality [0 <= 6, 1 sisanya] ","\n")
cat("----------------------------------------------------","\n")
df$target <- ifelse(df$quality <= 6,0,1)
head(df)

----------------------------------------- 
cek struktur data 
----------------------------------------- 
'data.frame':	1599 obs. of  12 variables:
 $ fixed.acidity       : num  7.4 7.8 7.8 11.2 7.4 7.4 7.9 7.3 7.8 7.5 ...
 $ volatile.acidity    : num  0.7 0.88 0.76 0.28 0.7 0.66 0.6 0.65 0.58 0.5 ...
 $ citric.acid         : num  0 0 0.04 0.56 0 0 0.06 0 0.02 0.36 ...
 $ residual.sugar      : num  1.9 2.6 2.3 1.9 1.9 1.8 1.6 1.2 2 6.1 ...
 $ chlorides           : num  0.076 0.098 0.092 0.075 0.076 0.075 0.069 0.065 0.073 0.071 ...
 $ free.sulfur.dioxide : num  11 25 15 17 11 13 15 15 9 17 ...
 $ total.sulfur.dioxide: num  34 67 54 60 34 40 59 21 18 102 ...
 $ density             : num  0.998 0.997 0.997 0.998 0.998 ...
 $ pH                  : num  3.51 3.2 3.26 3.16 3.51 3.51 3.3 3.39 3.36 3.35 ...
 $ sulphates           : num  0.56 0.68 0.65 0.58 0.56 0.56 0.46 0.47 0.57 0.8 ...
 $ alcohol             : num  9.4 9.8 9.8 9.8 9.4 9.4 9.4 10 9.5 10.5 ...
 $ quality             : int  5 

,fixed.acidity,volatile.acidity,citric.acid,residual.sugar,chlorides,free.sulfur.dioxide,total.sulfur.dioxide,density,pH,sulphates,alcohol,quality
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>
1,7.4,0.70,0.00,1.9,0.076,11,34,0.9978,3.51,0.56,9.4,5
2,7.8,0.88,0.00,2.6,0.098,25,67,0.9968,3.20,0.68,9.8,5
3,7.8,0.76,0.04,2.3,0.092,15,54,0.9970,3.26,0.65,9.8,5
4,11.2,0.28,0.56,1.9,0.075,17,60,0.9980,3.16,0.58,9.8,6
5,7.4,0.70,0.00,1.9,0.076,11,34,0.9978,3.51,0.56,9.4,5
6,7.4,0.66,0.00,1.8,0.075,13,40,0.9978,3.51,0.56,9.4,5


---------------------------------------------------- 
Membuat variabel target quality [0 <= 6, 1 sisanya]  
---------------------------------------------------- 


,fixed.acidity,volatile.acidity,citric.acid,residual.sugar,chlorides,free.sulfur.dioxide,total.sulfur.dioxide,density,pH,sulphates,alcohol,quality,target
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<dbl>
1,7.4,0.70,0.00,1.9,0.076,11,34,0.9978,3.51,0.56,9.4,5,0
2,7.8,0.88,0.00,2.6,0.098,25,67,0.9968,3.20,0.68,9.8,5,0
3,7.8,0.76,0.04,2.3,0.092,15,54,0.9970,3.26,0.65,9.8,5,0
4,11.2,0.28,0.56,1.9,0.075,17,60,0.9980,3.16,0.58,9.8,6,0
5,7.4,0.70,0.00,1.9,0.076,11,34,0.9978,3.51,0.56,9.4,5,0
6,7.4,0.66,0.00,1.8,0.075,13,40,0.9978,3.51,0.56,9.4,5,0


# Membangun model

In [29]:
# install pakege
if(!require(randomForest)){
  install.packages("randomForest")
  library(randomForest)
}

randomForest(as.factor(target)~.,
                       data = df,
                       ntree =10, # Jumlah pohon
                       mtry = 3) # jumlah variabel kandidat yang boleh dilihat pada setiap split node


Call:
 randomForest(formula = as.factor(target) ~ ., data = df, ntree = 10,      mtry = 3) 
               Type of random forest: classification
                     Number of trees: 10
No. of variables tried at each split: 3

        OOB estimate of  error rate: 0.89%
Confusion matrix:
     0   1 class.error
0 1351   6 0.004421518
1    8 208 0.037037037